# Task 2 — KDEF study

**Goal.** Retrain the classifiers of Task 1 on KDEF (frontal images only) using the feature representation
that worked best on FER2013, with a **subject-independent** evaluation, and compare the behaviour with FER2013.

**Protocol.**
* Only the straight/frontal pose (`S`); both photo sessions A and B of each person.
* Fixed 80/20 split **by subject** from `auxFiles/kdef_subject_split.pkl` (56 train / 14 test people):
  every image of a person is either in train or in test, never both.
* Hyperparameters are selected with **5-fold `GroupKFold` cross-validation grouped by subject** inside the
  training set (KDEF has no validation split; grouping prevents the A/B photos of one person from
  leaking between folds). Selection metric: macro-F1.
* The 14 test subjects are used once, for the final numbers.
* Images go through the provided `preprocess_kdef_image` (face detection → 20 % margin → grayscale →
  48×48) and are then rescaled to [0, 255] so that exactly the same feature extractors as for FER2013 can be used.
* KDEF labels are mapped to the FER2013 label ids (afraid → Fear, etc.).

Run from the `Assignment2_2026` folder after `task1_fer_baseline.ipynb`.

In [ ]:
import json, time, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             confusion_matrix, classification_report)

from src.preprocess import preprocess_kdef_image
from src.features import (EMOTIONS, KDEF_TO_FER, to_fer_scale, fer_pixels_to_images,
                          hog_features, landmark_features, pixel_features)

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 42
DATA = Path("data"); MODELS = Path("models"); RESULTS = Path("results")
FIGS = Path("figures/task2"); FIGS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(exist_ok=True); RESULTS.mkdir(exist_ok=True)

sel_path = RESULTS / "task1_selection.json"
task1_sel = json.load(open(sel_path)) if sel_path.exists() else {"best_feature": "HOG"}
BEST_FEATURE = task1_sel["best_feature"]
print("Best feature from Task 1 (FER2013 validation):", BEST_FEATURE)

## 1. Index the KDEF images and apply the subject split

In [ ]:
KDEF_EMO = {"AF": "afraid", "AN": "angry", "DI": "disgust", "HA": "happy",
            "NE": "neutral", "SA": "sad", "SU": "surprise"}
POSES = {"FL": "full_left", "HL": "half_left", "S": "straight", "HR": "half_right", "FR": "full_right"}

ROOT = DATA / "KDEF_and_AKDEF" / "KDEF"
records, skipped = [], []
for p in ROOT.rglob("*"):
    if p.suffix.lower() != ".jpg":
        continue
    stem = p.stem.upper()
    emo, pose = stem[4:6], stem[6:]
    if emo in KDEF_EMO and pose in POSES:
        records.append({"path": str(p), "acquisition": stem[0], "subject": stem[1:4],
                        "emotion": KDEF_EMO[emo], "pose": POSES[pose]})
    else:
        skipped.append(p.name)
print("skipped (unparseable file names):", skipped)

df = (pd.DataFrame(records)
        .sort_values(["acquisition", "subject", "emotion"]).reset_index(drop=True))
df["label"] = df.emotion.map(KDEF_TO_FER)
print(f"{len(df)} images, {df.subject.nunique()} subjects, sessions {sorted(df.acquisition.unique())}")

split = pickle.load(open("auxFiles/kdef_subject_split.pkl", "rb"))
train_subj, test_subj = set(split["train_subjects"]), set(split["test_subjects"])
assert not train_subj & test_subj, "subject overlap between train and test!"

front = df[df.pose == "straight"].copy()
front["split"] = np.where(front.subject.isin(train_subj), "train",
                          np.where(front.subject.isin(test_subj), "test", "unused"))
print(front.groupby("split").agg(images=("path", "size"), subjects=("subject", "nunique")))
pd.crosstab(front.emotion, front.split)

## 2. Preprocess to the FER2013 format
The provided `preprocess_kdef_image` returns floats in [0, 1]. FER2013 pixels are in [0, 255] and the
dlib landmark detector expects `uint8`, so the images are rescaled with `to_fer_scale` (×255) — without
this step the landmark detector receives an almost black image.
Images in which the Haar cascade finds no face are reported and dropped.

In [ ]:
cache = DATA / "kdef_preprocessed_fer_format.pkl"
if cache.exists():
    kd = pickle.load(open(cache, "rb"))
else:
    imgs, keep, failed = [], [], []
    for i, row in front.iterrows():
        try:
            imgs.append(to_fer_scale(preprocess_kdef_image(row.path)))
            keep.append(i)
        except Exception as e:
            failed.append((row.path, str(e)))
    kd = {"meta": front.loc[keep].reset_index(drop=True), "images": np.stack(imgs), "failed": failed}
    pickle.dump(kd, open(cache, "wb"))

meta, K_imgs = kd["meta"], kd["images"]
print(f"preprocessed: {len(K_imgs)} images, face detection failed for {len(kd['failed'])}:")
for f, e in kd["failed"]:
    print("   ", Path(f).name)

tr = (meta.split == "train").values; te = (meta.split == "test").values
y_tr, y_te = meta.label.values[tr], meta.label.values[te]
g_tr = meta.subject.values[tr]                       # groups for GroupKFold
print("train", tr.sum(), "test", te.sum())

In [ ]:
# KDEF after preprocessing vs FER2013 — the domain gap is visible already
fer_test = pd.read_csv(DATA / "fer2013_test.csv")
F_imgs = fer_pixels_to_images(fer_test.pixels)
F_y = fer_test.emotion.values

fig, axes = plt.subplots(2, 7, figsize=(14, 4.4))
for c in range(7):
    i = np.where(meta.label.values == c)[0][0]
    axes[0, c].imshow(K_imgs[i], cmap="gray", vmin=0, vmax=255); axes[0, c].set_title(f"KDEF {EMOTIONS[c]}", fontsize=9)
    j = np.where(F_y == c)[0][0]
    axes[1, c].imshow(F_imgs[j], cmap="gray", vmin=0, vmax=255); axes[1, c].set_title(f"FER {EMOTIONS[c]}", fontsize=9)
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.savefig(FIGS / "kdef_vs_fer_examples.png", dpi=150); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
axes[0].imshow(K_imgs.mean(0), cmap="gray"); axes[0].set_title("mean KDEF face")
axes[1].imshow(F_imgs.mean(0), cmap="gray"); axes[1].set_title("mean FER2013 face")
axes[2].hist(K_imgs.reshape(-1), bins=50, density=True, alpha=.6, label="KDEF")
axes[2].hist(F_imgs.reshape(-1), bins=50, density=True, alpha=.6, label="FER2013")
axes[2].legend(); axes[2].set_title("pixel intensity distribution")
for ax in axes[:2]: ax.axis("off")
plt.tight_layout(); plt.savefig(FIGS / "mean_faces.png", dpi=150); plt.show()

## 3. Features
The same extractors as in Task 1. For PCA the projection is fitted **inside** each cross-validation fold
(it is a step of the pipeline), keeping 95 % of the variance — the same rule as on FER2013.

The main experiment uses the best FER2013 feature (printed above). The other two features are also run,
because the dataset is small (cheap to compute) and it shows whether the *feature* ranking changes
between datasets as well — useful for Task 3.

In [ ]:
K_feats = {"HOG": hog_features(K_imgs), "Landmarks": landmark_features(K_imgs),
           "PCA": pixel_features(K_imgs)}   # PCA is applied inside the model pipeline
for k, v in K_feats.items(): print(k, v.shape)

## 4. Subject-grouped cross-validation for hyperparameters, then test on unseen subjects

In [ ]:
def make_model(feature, clf):
    steps = [PCA(n_components=0.95, random_state=RANDOM_STATE)] if feature == "PCA" else []
    if clf == "LogReg":
        steps += [StandardScaler(), LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)]
    elif clf == "SVM":
        steps += [StandardScaler(), SVC(kernel="rbf", gamma="scale", random_state=RANDOM_STATE)]
    else:
        steps += [RandomForestClassifier(n_estimators=300, max_features="sqrt", criterion="entropy",
                                         n_jobs=-1, random_state=RANDOM_STATE)]
    return make_pipeline(*steps)

GRIDS = {
    "LogReg": {"logisticregression__C": [0.001, 0.01, 0.1, 1, 10]},
    "SVM":    {"svc__C": [0.1, 0.3, 1, 3, 10, 100]},
    "RF":     {"randomforestclassifier__max_depth": [11, 20, None]},
}

def metrics(y, p):
    return {"accuracy": accuracy_score(y, p),
            "balanced_accuracy": balanced_accuracy_score(y, p),
            "macro_f1": f1_score(y, p, average="macro")}

cv = GroupKFold(n_splits=5)
rows, preds, fitted = [], {}, {}
for feat in [BEST_FEATURE] + [f for f in ["HOG", "Landmarks", "PCA"] if f != BEST_FEATURE]:
    X = K_feats[feat]
    for clf in ["LogReg", "SVM", "RF"]:
        t = time.time()
        gs = GridSearchCV(make_model(feat, clf), GRIDS[clf], scoring="f1_macro", cv=cv, n_jobs=-1)
        gs.fit(X[tr], y_tr, groups=g_tr)
        p = gs.predict(X[te])
        preds[(feat, clf)] = p; fitted[(feat, clf)] = gs.best_estimator_
        joblib.dump(gs.best_estimator_, MODELS / f"kdef_{feat.lower()}_{clf.lower()}.pkl")
        rows.append({"feature": feat, "classifier": clf,
                     "params": json.dumps({k.split("__")[1]: v for k, v in gs.best_params_.items()}),
                     "cv_macro_f1": gs.best_score_,
                     "cv_macro_f1_std": gs.cv_results_["std_test_score"][gs.best_index_],
                     **metrics(y_te, p)})
        print(f"{feat:9s} {clf:6s} CV macro-F1 {gs.best_score_:.3f}  test macro-F1 "
              f"{rows[-1]['macro_f1']:.3f}  {gs.best_params_}  ({time.time() - t:.0f}s)")

kres = pd.DataFrame(rows)
kres.to_csv(RESULTS / "task2_kdef_results.csv", index=False)

### Uncertainty of the test numbers
The test set has only 14 people (~196 images, ~28 per emotion), so one person more or less correct moves the
score noticeably. A 95 % bootstrap interval is computed by resampling **test subjects** (not images).

In [ ]:
subj_te = meta.subject.values[te]
uniq = np.unique(subj_te)
rng = np.random.default_rng(RANDOM_STATE)
boots = [np.concatenate([np.where(subj_te == s)[0] for s in rng.choice(uniq, len(uniq))]) for _ in range(1000)]

ci = []
for (feat, clf), p in preds.items():
    vals = [f1_score(y_te[b], p[b], average="macro") for b in boots]
    ci.append({"feature": feat, "classifier": clf,
               "macro_f1_ci_low": np.percentile(vals, 2.5), "macro_f1_ci_high": np.percentile(vals, 97.5)})
kres = kres.merge(pd.DataFrame(ci), on=["feature", "classifier"])
kres.to_csv(RESULTS / "task2_kdef_results.csv", index=False)

show = kres.set_index(["feature", "classifier"])[
    ["params", "cv_macro_f1", "cv_macro_f1_std", "accuracy", "balanced_accuracy", "macro_f1",
     "macro_f1_ci_low", "macro_f1_ci_high"]]
show.style.format({c: "{:.3f}" for c in show.columns if c != "params"}) \
          .background_gradient(cmap="Blues", subset=["macro_f1"])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
order = ["HOG", "Landmarks", "PCA"]
sns.barplot(data=kres, x="feature", y="macro_f1", hue="classifier", order=order,
            hue_order=["LogReg", "SVM", "RF"], ax=ax)
# bootstrap intervals on top of the bars
for bar_container, clf in zip(ax.containers, ["LogReg", "SVM", "RF"]):
    for bar, feat in zip(bar_container, order):
        r = kres[(kres.feature == feat) & (kres.classifier == clf)].iloc[0]
        x = bar.get_x() + bar.get_width() / 2
        ax.plot([x, x], [r.macro_f1_ci_low, r.macro_f1_ci_high], c="k", lw=1)
ax.set_ylim(0, 1); ax.axhline(1 / 7, ls=":", c="gray"); ax.set_title("KDEF test macro-F1 (95% subject bootstrap)")
plt.tight_layout(); plt.savefig(FIGS / "kdef_test_macro_f1.png", dpi=150); plt.show()

## 5. Confusion matrices (KDEF test subjects)

In [ ]:
def plot_cm(y, p, title, ax):
    cm = confusion_matrix(y, p, labels=range(7))
    cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    annot = np.array([[f"{cmn[i,j]:.2f}\n({cm[i,j]})" for j in range(7)] for i in range(7)])
    sns.heatmap(cmn, annot=annot, fmt="", cmap="Blues", vmin=0, vmax=1, cbar=False,
                xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=ax, annot_kws={"size": 7})
    ax.set_title(title, fontsize=10); ax.set_xlabel("predicted"); ax.set_ylabel("true")

feats_order = [BEST_FEATURE] + [f for f in order if f != BEST_FEATURE]
fig, axes = plt.subplots(3, 3, figsize=(18, 17))
for i, feat in enumerate(feats_order):
    for j, clf in enumerate(["LogReg", "SVM", "RF"]):
        plot_cm(y_te, preds[(feat, clf)], f"KDEF — {feat} + {clf}", axes[i, j])
plt.tight_layout(); plt.savefig(FIGS / "confusion_matrices_all.png", dpi=120); plt.show()

for (feat, clf), p in preds.items():
    f, ax = plt.subplots(figsize=(6, 5)); plot_cm(y_te, p, f"KDEF test — {feat} + {clf}", ax)
    plt.tight_layout(); f.savefig(FIGS / f"cm_{feat}_{clf}.png", dpi=150); plt.close(f)

## 6. Comparison with FER2013
* Does the **ranking of the classifiers** (and features) change between datasets?
* Which emotions are recognised well on both datasets?

FER2013 numbers are the PrivateTest results of Task 1.

In [ ]:
fer = pd.read_csv(RESULTS / "task1_results.csv").query("split == 'test'")
cmp = (fer[["feature", "classifier", "macro_f1", "balanced_accuracy"]]
         .merge(kres[["feature", "classifier", "macro_f1", "balanced_accuracy"]],
                on=["feature", "classifier"], suffixes=("_FER", "_KDEF")))
cmp["rank_FER"] = cmp.macro_f1_FER.rank(ascending=False).astype(int)
cmp["rank_KDEF"] = cmp.macro_f1_KDEF.rank(ascending=False).astype(int)
# classifier ranking *within* the best feature
bf = cmp[cmp.feature == BEST_FEATURE].copy()
bf["clf_rank_FER"] = bf.macro_f1_FER.rank(ascending=False).astype(int)
bf["clf_rank_KDEF"] = bf.macro_f1_KDEF.rank(ascending=False).astype(int)
cmp.to_csv(RESULTS / "task2_fer_vs_kdef.csv", index=False)
print(f"Classifier ranking with {BEST_FEATURE} features:")
display(bf[["classifier", "macro_f1_FER", "clf_rank_FER", "macro_f1_KDEF", "clf_rank_KDEF"]].round(3))
print("All 9 configurations (rank 1 = best macro-F1):")
cmp.sort_values("rank_KDEF").round(3)

In [ ]:
fer_pc = pd.read_csv(RESULTS / "task1_per_class_f1.csv", index_col=0)
rows = []
for clf in ["LogReg", "SVM", "RF"]:
    k = f1_score(y_te, preds[(BEST_FEATURE, clf)], average=None, labels=range(7))
    f = fer_pc.loc[f"{BEST_FEATURE} + {clf}"].values
    for e, kv, fv in zip(EMOTIONS, k, f):
        rows += [{"emotion": e, "classifier": clf, "dataset": "FER2013", "F1": fv},
                 {"emotion": e, "classifier": clf, "dataset": "KDEF", "F1": kv}]
pc = pd.DataFrame(rows)
pc.to_csv(RESULTS / "task2_per_class_f1_fer_vs_kdef.csv", index=False)

g = sns.catplot(data=pc, x="emotion", y="F1", hue="dataset", col="classifier", kind="bar",
                height=3.5, aspect=1.4, order=EMOTIONS)
g.set_xticklabels(rotation=30); g.fig.suptitle(f"Per-class F1 with {BEST_FEATURE} features", y=1.04)
g.savefig(FIGS / "per_class_f1_fer_vs_kdef.png", dpi=150); plt.show()

piv = pc.groupby(["emotion", "dataset"]).F1.mean().unstack().loc[EMOTIONS]
piv["KDEF - FER"] = piv.KDEF - piv.FER2013
piv.round(3)

## 7. Error analysis on KDEF

In [ ]:
best_row = kres[kres.feature == BEST_FEATURE].sort_values("cv_macro_f1", ascending=False).iloc[0]
bk = (best_row.feature, best_row.classifier)
print(f"Best {BEST_FEATURE} model by CV: {bk[1]}")
print(classification_report(y_te, preds[bk], target_names=EMOTIONS, digits=3))

p = preds[bk]; te_idx = np.where(te)[0]
wrong = np.where(p != y_te)[0]
n = min(16, len(wrong))
if n:
    fig, axes = plt.subplots(2, 8, figsize=(15, 4.4))
    for ax in axes.flat: ax.axis("off")
    for ax, w in zip(axes.flat, wrong[:n]):
        i = te_idx[w]
        ax.imshow(K_imgs[i], cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"{meta.subject[i]}{meta.acquisition[i]} T:{EMOTIONS[y_te[w]][:4]}\nP:{EMOTIONS[p[w]][:4]}",
                     fontsize=8, color="red")
    plt.suptitle(f"Misclassified KDEF test images ({len(wrong)}/{len(y_te)})")
    plt.tight_layout(); plt.savefig(FIGS / "kdef_misclassified.png", dpi=120); plt.show()

# errors per test subject: are some people systematically harder?
err_subj = pd.Series(p != y_te).groupby(subj_te).mean().sort_values(ascending=False)
err_subj.plot.bar(figsize=(8, 2.8), title="Error rate per KDEF test subject"); plt.tight_layout()
plt.savefig(FIGS / "kdef_error_per_subject.png", dpi=150); plt.show()

## 8. Observations (to be completed by the group)

* **Is KDEF easier or harder than FER2013, and why?** Think about: posed vs. spontaneous expressions,
  controlled lighting/background/pose, image quality, label reliability, class balance (KDEF is perfectly
  balanced), and on the other side the very small number of people (56 training subjects) and the
  wide CI on the test score.
* **Which models change ranking?** Use the ranking table in section 6. Relate it to sample size
  (~780 training images vs 28 709) and model capacity / variance.
* **Which emotions transfer well between datasets?** Use the per-class F1 comparison; which emotions are
  strong on both, which only on KDEF? Look at the confusion matrices for pairs that stay confused.
* Mention the face-detection failures and the effect of the 20 % margin crop (KDEF faces are smaller in the
  48×48 patch than in FER2013 — see the mean faces).